In [ ]:
import os
import glob
import re
import pandas as pd
#import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
# Path to folder with CSV files
data_path_68 = r"C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\monthly_data\data_o3\68"
data_path_46 = r"C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\monthly_data\data_o3\46"
data_path_32 = r"C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\monthly_data\data_o3\32"
data_path_22 = r"C:\Users\polgaste\Desktop\iasi_integration\mls_v6_test\monthly_data\data_o3\22"

In [ ]:
# Pattern to match files. Adjust if necessary (e.g. if you only want .csv files).
csv_files_68 = glob.glob(os.path.join(data_path_68, "monthly_*_*.csv"))
csv_files_46 = glob.glob(os.path.join(data_path_46, "monthly_*_*.csv"))
csv_files_32 = glob.glob(os.path.join(data_path_32, "monthly_*_*.csv"))
csv_files_22 = glob.glob(os.path.join(data_path_22, "monthly_*_*.csv"))

In [ ]:
# Prepare a list to gather all data
all_data_68 = []
all_data_46 = []
all_data_32 = []
all_data_22 = []

In [ ]:
# e.g., "monthly_nz_n_201268.csv" or "monthly_estonia_201268.csv"
# We'll capture everything between "monthly_" and the "_YYYY68.csv".
# Explanation:
#   ^monthly_      -> starts with "monthly_"
#   (.*?)          -> capture any characters, non-greedy, into group 1 (location)
#   _(\d{4})       -> underscore, then 4 digits for the year (group 2)
#   \d{2}          -> the trailing '68' (or other 2 digits)
#   \.csv$         -> .csv at the end
pattern = r"^monthly_(.*?)_(\d{4})\d{2}\.csv$"

In [ ]:
def averages_calculation(file_path, csv_files, all_data):
    for file_path in csv_files:
        filename = os.path.basename(file_path)
        match = re.match(pattern, filename)
        if not match:
            print(f"Skipping {filename}, doesn't match expected pattern.")
            continue

        location, year = match.group(1), match.group(2)

        # Read ragged rows manually, pad to equal length with NaN
        with open(file_path, 'r') as f:
            rows = [line.strip().split(',') for line in f if line.strip()]
        max_len = max(len(row) for row in rows)
        padded = [row + [float('nan')] * (max_len - len(row)) for row in rows]
        df = pd.DataFrame(padded).apply(pd.to_numeric, errors='coerce')

        monthly_stats = []
        for i in range(len(df)):
            row_data  = df.iloc[i].dropna()
            row_mean  = row_data.mean()
            row_std   = row_data.std()
            row_count = len(row_data)

            monthly_stats.append({
                "Month":              i + 1,
                "mean_concentration": row_mean,
                "std_concentration":  row_std,
                "count":              row_count,
            })

        monthly_df = pd.DataFrame(monthly_stats)
        monthly_df["location"] = location
        monthly_df["year"]     = int(year)

        all_data.append(monthly_df)
    return all_data

In [ ]:
averages_calculation(data_path_68, csv_files_68, all_data_68)


In [ ]:
averages_calculation(data_path_46, csv_files_46, all_data_46)


In [ ]:
averages_calculation(data_path_32, csv_files_32, all_data_32)


In [ ]:
averages_calculation(data_path_22, csv_files_22, all_data_22)


In [ ]:
# Concatenate all DataFrames into a single DataFrame
big_df_68 = pd.concat(all_data_68, ignore_index=True)
big_df_46 = pd.concat(all_data_46, ignore_index=True)
big_df_32 = pd.concat(all_data_32, ignore_index=True)
big_df_22 = pd.concat(all_data_22, ignore_index=True)

In [ ]:
big_df_68["date"] = pd.to_datetime({
    "year": big_df_68["year"].astype(int),
    "month": big_df_68["Month"].astype(int),
    "day": 1
})

big_df_46["date"] = pd.to_datetime({
    "year": big_df_46["year"].astype(int),
    "month": big_df_46["Month"].astype(int),
    "day": 1
})

big_df_32["date"] = pd.to_datetime({
    "year": big_df_32["year"].astype(int),
    "month": big_df_32["Month"].astype(int),
    "day": 1
})

big_df_22["date"] = pd.to_datetime({
    "year": big_df_22["year"].astype(int),
    "month": big_df_22["Month"].astype(int),
    "day": 1
})

In [ ]:
big_df_68

In [ ]:
#add latitude zones
#1 - north of 30°
#2 - between 30° and -30°
#3 - south of -30°
location_to_zone = {
    'bashkortostan': 1,
    'bozeman': 1,
    'brunei': 2,
    'california': 1,
    'catalonia': 1,
    'colombia': 2,
    'estonia': 1,
    'finland': 1,
    'florianopolis': 2,
    'florida': 2,
    'france': 1,
    'french_guiana': 2,
    'huntingdon': 1,
    'iceland_e': 1,
    'iceland_w': 1,
    'khabarovsk': 1,
    'kongo': 2,
    'kyrgyzstan': 1,
    'mexico': 2,
    'morocco': 1,
    'mukhrino': 1,
    'myanmar': 2,
    'nz_n': 3,
    'nz_s': 3,
    'pantanal': 2,
    'quistococha': 2,
    'romania': 1,
    'taiwan': 2,
    'tarapoto': 2,
    'tasmania': 3,
    'tierra_del_fuego': 3,
    'uganda_e': 2,
    'uganda_n': 2,
    'uganda_s': 2,
    'wales': 1
}

In [ ]:
big_df_68["latitude_zone"] = big_df_68["location"].map(location_to_zone)
big_df_46["latitude_zone"] = big_df_46["location"].map(location_to_zone)
big_df_32["latitude_zone"] = big_df_32["location"].map(location_to_zone)
big_df_22["latitude_zone"] = big_df_22["location"].map(location_to_zone)

In [ ]:
# Write out
big_df_68.to_csv("monthly_o3_68.csv", index=False)
big_df_46.to_csv("monthly_o3_46.csv", index=False)
big_df_32.to_csv("monthly_o3_32.csv", index=False)
big_df_22.to_csv("monthly_o3_22.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,8))
for loc in big_df_22["location"].unique():
    df_loc = big_df_22[big_df_22["location"] == loc].copy()
    # Sort by date (important if data is not already sorted)
    df_loc.sort_values("date", inplace=True)
    
    plt.plot(
        df_loc["date"], 
        df_loc["mean_concentration"], 
        marker='',
        linestyle='-',
        label=loc
    )

plt.xlabel("Date")
plt.ylabel("Mean O3 Concentration")
plt.title("Long Continuous Monthly Measurements by Location")
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(16,8))
for loc in big_df_68["location"].unique():
    df_loc = big_df_68[big_df_68["location"] == loc].copy()
    # Sort by date (important if data is not already sorted)
    df_loc.sort_values("date", inplace=True)
    
    plt.plot(
        df_loc["date"], 
        df_loc["mean_concentration"], 
        marker='',
        linestyle='-',
        label=loc
    )

plt.xlabel("Date")
plt.ylabel("Mean O3 Concentration")
plt.title("Long Continuous Monthly Measurements by Location")
plt.legend()
plt.show()
